In [1]:
import os
import shutil
import pydicom

# --- CONFIGURATION ---
source_path = '../Data/SQUARE/'

print(f"--- STARTING SMART CLEANING (SQUARE) ---")

def get_referenced_uid(rtstruct_path):
    try:
        dcm = pydicom.dcmread(rtstruct_path, stop_before_pixels=True, force=True)
        if 'ReferencedFrameOfReferenceSequence' in dcm:
            rfor = dcm.ReferencedFrameOfReferenceSequence[0]
            if 'RTReferencedStudySequence' in rfor:
                rstudy = rfor.RTReferencedStudySequence[0]
                if 'RTReferencedSeriesSequence' in rstudy:
                    return rstudy.RTReferencedSeriesSequence[0].SeriesInstanceUID
    except: return None
    return None

if os.path.exists(source_path):
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Found {len(patient_folders)} patients.")
    
    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        # 1. Find RTStruct
        rt_struct_path = None
        for root, dirs, files in os.walk(patient_dir):
            for f in files:
                if 'rs.' in f.lower() or (f.lower().startswith('rs') and '.dcm' in f.lower()):
                    rt_struct_path = os.path.join(root, f)
                    break
            if rt_struct_path: break
            
        if not rt_struct_path:
            print(f"[{i+1}] {patient_id}: [SKIP] No RTStruct found")
            continue

        # 2. Get Target UID
        target_uid = get_referenced_uid(rt_struct_path)
        if not target_uid:
            print(f"[{i+1}] {patient_id}: [SKIP] RTStruct has no UID")
            continue
            
        # 3. Create Clean Folders
        ct_dir = os.path.join(patient_dir, 'CT')
        struct_dir = os.path.join(patient_dir, 'Struct')
        
        if not os.path.exists(ct_dir): os.makedirs(ct_dir)
        if not os.path.exists(struct_dir): os.makedirs(struct_dir)
        
        # 4. Move Matching Files
        moved_count = 0
        for root, dirs, files in os.walk(patient_dir):
            # Don't scan the new folders we just made!
            if 'CT' in root or 'Struct' in root: continue
            
            for f in files:
                full_path = os.path.join(root, f)
                try:
                    dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    
                    # If it matches the UID -> Move to CT
                    if dcm.SeriesInstanceUID == target_uid:
                        shutil.move(full_path, os.path.join(ct_dir, f))
                        moved_count += 1
                        
                    # If it is the RTStruct -> Move to Struct
                    elif full_path == rt_struct_path:
                        shutil.move(full_path, os.path.join(struct_dir, f))
                        
                except: continue
        
        print(f"[{i+1}] {patient_id}: Organized {moved_count} CT files.")

    print("\nSUCCESS! Square dataset cleaned.")
else:
    print("Source path not found.")

--- STARTING SMART CLEANING (SQUARE) ---
Found 22 patients.
[1] R1508007367: Organized 168 CT files.
[2] R1710009613: Organized 200 CT files.
[3] R1803003706: Organized 176 CT files.
[4] R2104010433: Organized 160 CT files.
[5] R2109007606: Organized 207 CT files.
[6] R2207007065: Organized 160 CT files.
[7] R2208002785: Organized 192 CT files.
[8] R2208006289: Organized 192 CT files.
[9] R2209009915: Organized 152 CT files.
[10] R2210004015: Organized 160 CT files.
[11] R2212004694: Organized 208 CT files.
[12] R2212008104: Organized 167 CT files.
[13] R2307002820: Organized 164 CT files.
[14] R2307010772: Organized 164 CT files.
[15] R2312007858: Organized 200 CT files.
[16] R2405000429: Organized 192 CT files.
[17] R2405006441: Organized 208 CT files.
[18] R2407004094: Organized 168 CT files.
[19] R2412007965: Organized 231 CT files.
[20] R2503001547: Organized 160 CT files.
[21] R2506009982: Organized 184 CT files.
[22] R2506010421: Organized 177 CT files.

SUCCESS! Square dataset 